Accompanying Code for: <br>
**Johannes Zeitler, Meinard Müller. "A Unified Perspective on CTC and SDTW using Differentiable DTW", submitted to IEEE Transactions on Audio, Speech, and Language Processing, 2025.**

Johannes Zeitler (johannes.zeitler@audiolabs-erlangen.de), 2025

### Description
Compute and/or display evaluation metrics: Frame-wise accuracy, retrieval performance, sequence decoding and label error rate

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch.nn as nn
import torch.nn.functional as F
import librosa
from IPython.display import display, Audio, HTML
from libfmp import b
from hcqt import compute_hopsize_cqt

/home/jzeitler@alabsad.fau.de/anaconda3/envs/dDTW_CTC_fixedVersions/lib/python3.11/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
model_path = "./saved_models"
log_path = "./logs"

configs = []

# horizontal step weight (1-0) = 1.0
config = {"ID": "CTC-A",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1.0}
configs.append(config)

config = {"ID": "CTC-B",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":2.0}
configs.append(config)

config = {"ID": "CTC-C",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1e20}
configs.append(config)

config = {"ID": "SDTW-D",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [1,1]],
          "step_weights": [1., 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)

config = {"ID": "SDTW-E",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [0,1], [1,1]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)

#########################################################
# horizontal step weight (1-0) = 0.1

config = {"ID": "CTC-A-W",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1.0}
configs.append(config)

config = {"ID": "CTC-B-W",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":2.0}
configs.append(config)

config = {"ID": "CTC-C-W",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1e20}
configs.append(config)

config = {"ID": "SDTW-D-W",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [1,1]],
          "step_weights": [0.1, 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)

config = {"ID": "SDTW-E-W",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [0,1], [1,1]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)
###############################################



# EM
config = {"ID": "EM",
          "loss_variant": "EM",
          "step_sizes": [[1,0], [0,1], [1,1]],
          "step_weights": [1.0, 1.0, 1.5],
          "gamma": 0,
          "min_function": "hardmin"}
configs.append(config)

###################################################

# strong
config = {"ID": "strong",
          "loss_variant": "strong",
          "step_sizes": [],
          "step_weights": [],
          "gamma": 0,
          "min_function": ""}
configs.append(config)

In [3]:
# check if models for all configs are available
n_runs = 5
for config in configs:
    for r in range(n_runs):        
        exp_name = config["ID"] + "_run%i"%(r)
        if not os.path.isfile(os.path.join(model_path, "model_%s_bestVal.pt"%(exp_name))):
            print(exp_name)

### set up test split and dataset

In [4]:
def get_files_for_split(path, idx):
    files = []
    for i in idx:
        split_in = pd.read_csv(os.path.join(path, "MTD_split5-%i.csv"%(i)))

        for _, row in split_in.iterrows():
            filename = row[0].split(".")[0]
            if os.path.isfile(os.path.join("data", "hcqt", filename+".npy")):
                files.append(filename)
    return files

In [5]:
split_dir = (os.path.join("data", "MTD_splits"))
splits_test = [5]
files_test = get_files_for_split(split_dir, splits_test)

In [6]:
device_preload = "cuda:0"

class MTD_Dataset(torch.utils.data.Dataset):
    def __init__(self, file_list, device_preload="cuda", device_out="cuda"):
        super().__init__()

        self.file_list = file_list
        self.device_preload=device_preload
        self.device_out=device_out
        self.hcqt_in = []
        self.strong_targets_in = []
        self.weak_targets_in = []
        self.CTC_targets_in = []
        
        for file in tqdm(file_list):
            hcqt = np.load(os.path.join("data", "hcqt", file+".npy")).transpose(1,0,2)    
            self.hcqt_in.append(torch.tensor(hcqt, device=device_preload, dtype=torch.float32))
        
            strong_targets = np.load(os.path.join("data", "strong_targets_chroma", file+".npy")).transpose(1,0)
            self.strong_targets_in.append(torch.tensor(strong_targets, device=device_preload, dtype=torch.float32))
        
            weak_targets = np.load(os.path.join("data", "weak_targets_chroma", file+".npy")).transpose(1,0)
            self.weak_targets_in.append(torch.tensor(weak_targets, device=device_preload, dtype=torch.float32))
        
            CTC_targets = np.load(os.path.join("data", "CTC_targets_chroma", file+".npy"))
            self.CTC_targets_in.append(torch.tensor(CTC_targets, device=device_preload, dtype=torch.int8))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        return {"file": self.file_list[idx],
                "hcqt": self.hcqt_in[idx].to(self.device_out), 
                "strong_targets": self.strong_targets_in[idx].to(self.device_out), 
                "weak_targets": self.weak_targets_in[idx].to(self.device_out), 
                "CTC_targets": self.CTC_targets_in[idx].to(self.device_out)
               }

In [7]:
dataset_test = MTD_Dataset(file_list = files_test)

def weak_collate(batch):
    result = {}
    result["file"] = [b["file"] for b in batch]   
    
    for key in batch[0].keys():
        if key == "file": 
            continue
        else:
            result[key] = torch.nn.utils.rnn.pad_sequence([b[key] for b in batch], batch_first=True)
            result[key+"_lengths"] = [b[key].shape[0] for b in batch]

    return result

test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=8, shuffle=False, drop_last=False, collate_fn=weak_collate, num_workers=0)

  0%|          | 0/2 [00:00<?, ?it/s]

### define the model

In [8]:
class ThemeEnhancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=6, out_channels=64, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv4 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv5 = nn.Conv2d(in_channels=32, out_channels=8, kernel_size=(3,42), stride=(1,1), padding="same")
        self.conv6 = nn.Conv2d(in_channels=8, out_channels=1, kernel_size=(1,1), stride=(1,1), padding="same")

        self.pool1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1,3), stride=(1,3), padding="valid")
        self.pool1.weight = nn.Parameter(torch.ones(1,1,1,3))
        self.pool1.bias = nn.Parameter(torch.tensor([0.]))

        self.pool2 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1,61), stride=(1,1), padding="valid")
        pool2_weights = torch.zeros(1,1,1,61)
        for i in range(61):
            if (i%12)==0:
                pool2_weights[0,0,0,i] = 1
        self.pool2.weight = nn.Parameter(pool2_weights)
        self.pool2.bias = nn.Parameter(torch.tensor([0.]))

        self.pool1.weight.requires_grad=False
        self.pool1.bias.requires_grad=False
        self.pool2.weight.requires_grad=False
        self.pool2.bias.requires_grad=False

        self.convEps = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1,216), stride=(1,1), padding="valid")
    
        

    def forward(self, x):

        column_norm = torch.clamp(torch.sum(x**2, axis=-1, keepdims=True), min=1e-10)
        y = x / column_norm
        
        y = F.leaky_relu(self.conv1(y))
        y = F.leaky_relu(self.conv2(y))
        y = F.leaky_relu(self.conv3(y))
        y = F.leaky_relu(self.conv4(y))
        y = F.leaky_relu(self.conv5(y))
        y = F.sigmoid(self.conv6(y))

        out_chroma = self.pool1(y)
        pred_chroma = self.pool2(out_chroma)
        eps = self.convEps(y)
        pred_CTC = torch.cat([pred_chroma, eps], axis=-1)
        
        return pred_chroma[:,0,:,:], pred_CTC[:,0,:,:], F.softmax(pred_chroma[:,0,:,:], dim=-1), F.softmax(pred_CTC[:,0,:,:], dim=-1)  


In [9]:
model = ThemeEnhancer().to("cuda")

In [10]:
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
print(params)

72970


In [11]:
from scipy.ndimage import median_filter, convolve
smooth_filt = np.ones((5,1))/5

### Evaluation

In [12]:
# label error rate (similar to word/character error rate used in ASR)
def label_error_rate_detailed(ref, hyp):
    """
    Compute Label Error Rate (LER) and individual error rates.
    Args:
        ref (list[str] or list[int]): reference label sequence
        hyp (list[str] or list[int]): hypothesis label sequence
    Returns:
        ler (float): total label error rate = (S + D + I) / len(ref)
        s_rate (float): substitutions / len(ref)
        d_rate (float): deletions / len(ref)
        i_rate (float): insertions / len(ref)
    """
    n = len(ref)
    m = len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    backtrace = [[None] * (m + 1) for _ in range(n + 1)]

    # Initialize first row/column
    for i in range(1, n + 1):
        dp[i][0] = i
        backtrace[i][0] = 'D'
    for j in range(1, m + 1):
        dp[0][j] = j
        backtrace[0][j] = 'I'

    # Fill DP table
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
                backtrace[i][j] = 'OK'
            else:
                # Try each operation
                deletion = dp[i - 1][j] + 1
                insertion = dp[i][j - 1] + 1
                substitution = dp[i - 1][j - 1] + 1

                dp[i][j], op = min(
                    (deletion, 'D'),
                    (insertion, 'I'),
                    (substitution, 'S'),
                    key=lambda x: x[0]
                )
                backtrace[i][j] = op

    # Backtrack from bottom-right corner
    i, j = n, m
    S = D = I = 0
    while i > 0 or j > 0:
        op = backtrace[i][j]

        if op == 'S':
            S += 1
            i -= 1
            j -= 1
        elif op == 'D':
            D += 1
            i -= 1
        elif op == 'I':
            I += 1
            j -= 1
        elif op == 'OK':
            i -= 1
            j -= 1
        else:
            # Should not happen, but for safety:
            break

    ler = (S + D + I) / n
    return ler, S / n, D / n, I / n
    # return LER, #substitutions, #deletions, #insertions

from collections import defaultdict

# standard beam search
def ctc_beam_search(probs, beam_width=10, blank_id=None, idx2char=None):
    """
    Perform prefix beam search decoding for CTC output probabilities.

    Args:
        probs: Tensor of shape (T, D) — probabilities (not log) over D symbols at T time steps.
        beam_width: Number of active hypotheses to keep.
        blank_id: Index of the blank symbol in the vocabulary.
        idx2char: Optional list mapping class indices to characters.

    Returns:
        best_sequence (str): decoded string or tuple of indices.
    """
    if probs.ndim == 3:
        probs = probs.squeeze(0)  # (T, D)
    T, D = probs.shape

    # Initialize the beam with the empty prefix (as tuple)
    Beam = {(): (1.0, 0.0)}  # prefix -> (p_blank, p_nonblank)

    for t in range(T):
        new_beam = defaultdict(lambda: (0.0, 0.0))
        for prefix, (p_b, p_nb) in Beam.items():
            for c in range(D):
                p = probs[t, c].item()

                if c == blank_id:
                    # Stay with same prefix, now ending with blank
                    p_b_new, p_nb_new = new_beam[prefix]
                    p_b_new += (p_b + p_nb) * p
                    new_beam[prefix] = (p_b_new, p_nb_new)
                else:
                    # Extend the prefix with symbol c
                    end_char = prefix[-1] if prefix else None
                    new_prefix = prefix + (c,)

                    p_b_new, p_nb_new = new_beam[new_prefix]
                    if c == end_char:
                        p_nb_new += p_b * p
                    else:
                        p_nb_new += (p_b + p_nb) * p
                    new_beam[new_prefix] = (p_b_new, p_nb_new)

                    # If char repeats, also allow staying at same prefix
                    if c == end_char:
                        p_b_old, p_nb_old = new_beam[prefix]
                        p_nb_old += p_nb * p
                        new_beam[prefix] = (p_b_old, p_nb_old)

        # Prune beam to top-K
        Beam = dict(
            sorted(
                new_beam.items(),
                key=lambda kv: kv[1][0] + kv[1][1],
                reverse=True
            )[:beam_width]
        )

    # Pick best prefix (tuple of indices)
    best_prefix, (p_b, p_nb) = max(Beam.items(), key=lambda kv: kv[1][0] + kv[1][1])

    # Convert to string if idx2char is provided
    if idx2char:
        best_seq = ''.join(idx2char[i] for i in best_prefix)
    else:
        best_seq = best_prefix  # return tuple of indices if no mapping provided

    return best_seq

### evaluate and plot

In [ ]:
all_results = {}

n_runs = 5
for config in configs:
    for r in range(n_runs):        
        exp_name = config["ID"]+"_run%i"%(r)        
        if not os.path.isfile(os.path.join(model_path, "model_%s_bestVal.pt"%(exp_name))):
            print(exp_name)

        # load a model
        print("evaluating %s"%(exp_name))
        model.load_state_dict(torch.load(os.path.join(model_path, "model_%s_bestVal.pt"%(exp_name)), weights_only=True))
        model.eval()

        edit_distances = [[], [], [], []]
        edit_distances_beamSearch = [[], [], [], []]
        matches_discr = []

        # compute metrics
        for batch in test_loader:
            hcqt = batch["hcqt"].permute(0,3,1,2)
            seq_lengths = batch["hcqt_lengths"]
            strong_targets = batch["strong_targets"]
            strong_target_lengths = batch["strong_targets_lengths"]
            target_lengths = batch["CTC_targets_lengths"]
            CTC_targets = batch["CTC_targets"]
        
            chroma_raw, CTC_raw, chroma_prob, CTC_prob = model(hcqt)
        
            for ib in range(hcqt.shape[0]):
                N = seq_lengths[ib]

                # frame-wise pitch discrete pitch class accuracy
                match_disc = torch.mean( (chroma_prob[ib,:N].argmax(axis=-1) == strong_targets[ib,:N].argmax(axis=-1))*1.)            
                matches_discr.append(match_disc.item())
    
                strong_targets_batch = strong_targets[ib,:seq_lengths[ib]].detach().cpu().numpy()
                chroma_probs_batch = chroma_prob[ib,:seq_lengths[ib]].detach().cpu().numpy()
                CTC_probs_batch = CTC_prob[ib, :seq_lengths[ib]].detach().cpu().numpy()
    
                ref_seq = CTC_targets[ib, :target_lengths[ib]]
    
                # beam search with blank
                est_seq_decoded = ctc_beam_search(CTC_probs_batch, beam_width=10, blank_id=12)#
                ler, Sr, Dr, Ir = label_error_rate_detailed(ref_seq, est_seq_decoded)                
                edit_distances_beamSearch[0].append(ler)
                edit_distances_beamSearch[1].append(Sr)
                edit_distances_beamSearch[2].append(Dr)
                edit_distances_beamSearch[3].append(Ir)
    
                # greedy edit distances
                am = CTC_probs_batch.argmax(axis=-1)
                # drop duplicates
                diffs = np.abs(am[:-1] - am[1:]) > 0
                diffs = np.concatenate([np.array([True]), diffs])
                est_seq = am[diffs]
                # drop blanks
                est_seq = est_seq[est_seq < 12]           
                ler, Sr, Dr, Ir = label_error_rate_detailed(ref_seq, est_seq)
                    
                edit_distances[0].append(ler)
                edit_distances[1].append(Sr)
                edit_distances[2].append(Dr)
                edit_distances[3].append(Ir)

        all_results[exp_name] = {"edit_distances": edit_distances,
                                 "edit_distances_beamSearch": edit_distances_beamSearch,
                                 "matches_discr": matches_discr}

### plot results

In [ ]:
strings_out = []
best_models_val = []

strings_out.append("Config \t\tBinary Acc.\t\tCont. Acc.\t\t CER\t\t\tCER smoothed\t\tCER beam search")

strings_table = []
strings_table.append("Config \t\tBinary Acc.\t\t\t\t LER\t\t\tCER beam search")

IDs = []

fig, ax = plt.subplots(3,1, figsize=(10,15))

n_runs = 5
for i_config, config in enumerate(configs):

    matches_discr = []
    matches_cont = []
    edit_distances = []
    edit_distances_smoothed = []
    edit_distances_beamSearch = []
    val_losses = []

    IDs.append(config["ID"])

    exp_names = []
    
    for r in range(n_runs):
        
        exp_name = config["ID"] + "_run%i"%(r)

        exp_names.append(exp_name)

        log_in = pd.read_csv(os.path.join(log_path, "%s_stats.csv"%(exp_name)), sep=";")
        val_losses.append(np.min(log_in.val_loss))

        res = all_results[exp_name]

        
        matches_discr.append(np.mean(res["matches_discr"]))
        edit_distances.append(np.mean(res["edit_distances"]))

        edit_distances_beamSearch.append(np.mean(res["edit_distances_beamSearch"]))
        
        ax[0].scatter(i_config, np.mean(res["matches_discr"]), marker="x", color="tab:blue")
        ax[1].scatter(i_config, np.mean(res["edit_distances"]), marker="x", color="tab:blue")
        ax[2].scatter(i_config, np.mean(res["edit_distances_beamSearch"]), marker="x", color="tab:blue")
        
    
    ax[0].scatter(i_config, np.mean(matches_discr), marker="_", color="tab:red", s=50)
    ax[0].scatter(i_config, np.median(matches_discr), marker="x", color="tab:red", s=50)
    

    ax[1].scatter(i_config, np.mean(edit_distances), marker="_", color="tab:red", s=50)
    ax[1].scatter(i_config, np.median(edit_distances), marker="x", color="tab:red", s=50)


    ax[2].scatter(i_config, np.mean(edit_distances_beamSearch), marker="_", color="tab:red", s=50)
    ax[2].scatter(i_config, np.median(edit_distances_beamSearch), marker="x", color="tab:red", s=50)

    bestValRun = int(np.argmin(val_losses))

    config["best_val_run"] = bestValRun

    best_models_val.append(exp_names[bestValRun])


    strings_table.append("%s & \t$%.3f$ & $%.3f$ &\t$%.3f$ & $%.3f$ &\t$%.3f$ & $%.3f$\\\\"%(config["ID"].ljust(10), 
                                                                              np.mean(matches_discr), np.max(matches_discr),
                                                                              np.mean(edit_distances), np.min(edit_distances), 
                                                                              np.mean(edit_distances_beamSearch), np.min(edit_distances_beamSearch)))
    

ax[0].set_title("Binarized Chroma Accuracy")
ax[1].set_title("Edit Distance")
ax[2].set_title("Edit Distance with Beam Search")
    
for a in ax:
    a.set_xticks(np.arange(len(IDs)))
    a.set_xticklabels(IDs, fontsize=8)
    a.grid()

fig.tight_layout()
plt.show()


print()
for msg in strings_table:
    print(msg)

print()
print("Models with lowest validation loss:")
for exp_name in best_models_val:
    print("\"model_%s_bestVal.pt\","%(exp_name))

### plot retrieval results

In [ ]:
colors=["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:brown", "magenta", "cyan", "lime", "tab:purple", "lightblue", "green", "red", "black"]
top_k = np.arange(1,50)

fig, ax = plt.subplots(1,1, figsize=(8,5))
for config, color in zip(configs, colors):
    data_in = np.load("./results/matchings/"+config["ID"]+".npz", allow_pickle=True)
    ratings = data_in["ratings"]#(rank(min_scores/wp_lengths, axis=1) * query_db_match).sum(axis=-1)    plot_list = [np.mean(ratings < k) for k in top_k]
    plot_list = [np.mean(ratings < k) for k in top_k]
    ax.plot(top_k, plot_list, linewidth=2, color=color, linestyle="-", label = config["ID"])

ax.grid()
ax.legend(loc="lower right")

ax.set_xlabel("Top k")
ax.set_ylabel("Retrieval probability")

ax.set_title("Top k retrieval probability")
fig.tight_layout()
plt.show()

In [16]:
# print a table

top_k = [1, 5, 10, 25, 50]

print("Loss    \t" + "".join(["Top %i\t"%(k) for k in top_k])+"MRR")

table_str = []
table_str.append("Loss       &\t" + "".join(["Top %i & \t"%(k) for k in top_k])+"MRR \\\\")

for config in configs:
    data_in = np.load("./results/matchings/"+config["ID"]+".npz", allow_pickle=True)
    ratings = data_in["ratings"]
    plot_list = [np.mean(ratings < k) for k in top_k]

    MRR = np.mean(1/(1+ratings))

    out_str = "%s\t"%(config["ID"].ljust(10)) + "".join(["%.3f\t"%(r) for r in plot_list]) + "%.3f"%(MRR)

    table_str.append("%s &\t"%(config["ID"].ljust(10)) + "".join(["$%.3f$ &\t"%(r) for r in plot_list]) + "$%.3f$"%(MRR) + " \\\\")
    print(out_str)

print()
print()
for msg in table_str:
    print(msg)

Loss    	Top 1	Top 5	Top 10	Top 25	Top 50	MRR
CTC-A     	0.000	0.500	1.000	1.000	1.000	0.217
CTC-B     	0.500	0.500	1.000	1.000	1.000	0.571
CTC-C     	0.000	0.500	1.000	1.000	1.000	0.306
SDTW-D    	0.000	0.500	1.000	1.000	1.000	0.175
SDTW-E    	0.000	0.500	1.000	1.000	1.000	0.217
CTC-A-W   	0.000	0.500	1.000	1.000	1.000	0.196
CTC-B-W   	0.000	0.500	1.000	1.000	1.000	0.175
CTC-C-W   	0.000	0.500	1.000	1.000	1.000	0.150
SDTW-D-W  	0.500	0.500	1.000	1.000	1.000	0.550
SDTW-E-W  	0.000	0.500	1.000	1.000	1.000	0.300
EM        	0.500	0.500	1.000	1.000	1.000	0.571
strong    	0.000	0.000	1.000	1.000	1.000	0.113


Loss       &	Top 1 & 	Top 5 & 	Top 10 & 	Top 25 & 	Top 50 & 	MRR \\
CTC-A      &	$0.000$ &	$0.500$ &	$1.000$ &	$1.000$ &	$1.000$ &	$0.217$ \\
CTC-B      &	$0.500$ &	$0.500$ &	$1.000$ &	$1.000$ &	$1.000$ &	$0.571$ \\
CTC-C      &	$0.000$ &	$0.500$ &	$1.000$ &	$1.000$ &	$1.000$ &	$0.306$ \\
SDTW-D     &	$0.000$ &	$0.500$ &	$1.000$ &	$1.000$ &	$1.000$ &	$0.175$ \\
SDTW-E     &	$0.000$ &	$

### Plots for running example in paper

In [ ]:
MTDID = "0859"
metadata = pd.read_csv("./data/MTD.csv", sep=";")
filename_MTD = metadata[metadata.MTDID==MTDID].WAV_MTD.item().split(".")[0]
end_frame = 86

In [ ]:
dataset_test = MTD_Dataset(file_list = [filename_MTD])
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=8, shuffle=False, drop_last=False, collate_fn=weak_collate, num_workers=0)

for batch in test_loader:
    continue

pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B", "<blank>"]
len(pitch_classes)

In [21]:
def bring_blank_down(x, feature_dim=0):
    x_out = np.zeros_like(x)

    if feature_dim == 0:
        x_out[0] = x[-1]
        x_out[1:] = x[:-1]
    
    elif feature_dim == 1:
        x_out[:,0] = x[:,-1]
        x_out[:,1:] = x[:,:-1]

    return x_out

In [ ]:
figOut, axOut = plt.subplots(1,1, figsize=(5,2))

fig, ax = plt.subplots(1+len(configs),2, figsize=(10,20))

ax[0,0].imshow(batch["strong_targets"][0,:end_frame].T.detach().cpu().numpy(), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")
ax[0,1].imshow(batch["strong_targets"][0,:end_frame].T.detach().cpu().numpy(), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")

axOut.imshow(batch["strong_targets"][0,:end_frame].T.detach().cpu().numpy(), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")
figOut.tight_layout()
axOut.set_axis_off()
figOut.savefig(os.path.join("figures", "runEx_MTD_strongTargets.png"),
            dpi=250, bbox_inches="tight", pad_inches=0)

for i_model, config in enumerate(configs):
    model.load_state_dict(torch.load(os.path.join(model_path, "model_%s_run%i_bestVal.pt"%(config["ID"], config["best_val_run"])), weights_only=True))
    model.eval()

    model_ID = config["ID"]

    for batch in test_loader:
        hcqt = batch["hcqt"].permute(0,3,1,2)
        seq_lengths = batch["hcqt_lengths"]
        strong_targets = batch["strong_targets"]
        strong_target_lengths = batch["strong_targets_lengths"]
        target_lengths = batch["CTC_targets_lengths"]
        CTC_targets = batch["CTC_targets"]
    
        chroma_raw, CTC_raw, chroma_prob, CTC_prob = model(hcqt)

        ######################################################
        ib = 0
        
        N = seq_lengths[ib]
    
        match_disc = torch.mean( (chroma_prob[ib,:N].argmax(axis=-1) == strong_targets[ib,:N].argmax(axis=-1))*1.)
        match_cont = torch.mean(torch.sum( chroma_prob[ib,:N] * strong_targets[ib,:N], axis=-1))
    

        strong_targets_batch = strong_targets[ib,:seq_lengths[ib]].detach().cpu().numpy()
        chroma_probs_batch = chroma_prob[ib,:seq_lengths[ib]].detach().cpu().numpy()
        CTC_probs_batch = CTC_prob[ib, :seq_lengths[ib]].detach().cpu().numpy()

        ref_seq = CTC_targets[ib, :target_lengths[ib]]

        # beam search with blank
        est_seq_decoded = ctc_beam_search(CTC_probs_batch, beam_width=10, blank_id=12)#
        ler_beam, Sr, Dr, Ir = label_error_rate_detailed(ref_seq, est_seq_decoded)                

        

        # greedy decoding
        am = CTC_probs_batch.argmax(axis=-1)
        diffs = np.abs(am[:-1] - am[1:]) > 0
        diffs = np.concatenate([np.array([True]), diffs])
        est_seq_greedy = am[diffs]
        est_seq_greedy = est_seq_greedy[est_seq_greedy < 12]        
        ler_greedy, Sr, Dr, Ir = label_error_rate_detailed(ref_seq, est_seq_greedy)

        

    print(model_ID)
    print("Accuracy: %.3f"%(match_disc))
    print("Beam search decoding: LER = %.3f"%(ler_beam))
    print(", ".join([pitch_classes[p] for p in est_seq_decoded]))
    print()
    print("Greedy decoding: LER = %.3f"%(ler_greedy))
    print(", ".join([pitch_classes[p] for p in est_seq_greedy]))
    print("------------------------------------------------------")
    print()

    ax[i_model+1,0].imshow(bring_blank_down(CTC_prob[0,:end_frame].T.detach().cpu().numpy()), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")
    ax[i_model+1,1].imshow(chroma_prob[0,:end_frame].T.detach().cpu().numpy(), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")
    ax[i_model+1,0].set_title(model_ID)


    # save with blank
    axOut.imshow(bring_blank_down(CTC_prob[0,:end_frame].T.detach().cpu().numpy()), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")
    figOut.tight_layout()
    axOut.set_axis_off()
    figOut.savefig(os.path.join("figures", "runEx_MTD_%s_blank.png"%(model_ID)),
                dpi=250, bbox_inches="tight", pad_inches=0)

    # save without blank
    axOut.imshow(chroma_prob[0,:end_frame].T.detach().cpu().numpy(), aspect="auto", origin="lower", interpolation="None", cmap="gray_r")
    figOut.tight_layout()
    axOut.set_axis_off()
    figOut.savefig(os.path.join("figures", "runEx_MTD_%s.png"%(model_ID)),
                dpi=250, bbox_inches="tight", pad_inches=0)

fig.tight_layout()
plt.show()